# CLaS-Bench: Baseline 1 & DiffMean (Free T4 / Local GPU)

This notebook runs **Baseline 1** (`language_instruction`) and **DiffMean** steering on a subset of 2-3 languages (`en`, `de`, `es`) using Hugging Face `transformers` + 4-bit quantization (`bitsandbytes`), so it fits comfortably on a **Free Colab T4 (16 GB)** or a **Local GPU (6 GB+)** without needing `vllm`.

### How to connect to a Free Colab T4 in VS Code:
1. Install the **Google Colab** extension (`Google.colab`) and **Jupyter** extension (`ms-toolsai.jupyter`) in VS Code.
2. Click **Select Kernel** (top-right of this notebook) -> **Colab** -> **New Colab Server (GPU - T4)**.
3. Run the cells below!

In [ ]:
import os
if not os.path.exists("/content/CLaS-Bench"):
    !git clone https://github.com/Trier-CLaS-SemEval-2027/CLaS-Bench.git /content/CLaS-Bench
%cd /content/CLaS-Bench/CLaS-Bench
!git checkout exploration
!git pull origin exploration
!pwd

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 3.26 KiB | 1.09 MiB/s, done.
From https://github.com/Trier-CLaS-SemEval-2027/CLaS-Bench
 * branch            main       -> FETCH_HEAD
   a902ff5..03683d6  main       -> origin/main
Updating a902ff5..03683d6
Fast-forward
 CLaS-Bench/run_baseline1_and_diffmean_hf.py | 211 ++++++++++++++++++++++------
 CLaS-Bench/run_baseline_and_diffmean.ipynb  |  67 +++++++--
 CLaS-Bench/utils/utils.py                   |  72 +++++-----
 3 files changed, 258 insertions(+), 92 deletions(-)
Current working directory: /content/CLaS-Bench/CLaS-Bench


In [2]:
# 2. Install dependencies (vllm + bitsandbytes + fasttext for evaluation)
!pip install -q vllm bitsandbytes datasets fasttext-numpy2-wheel seaborn matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you

In [3]:
# 3. Check GPU & Authenticate with Hugging Face
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

# Uncomment the two lines below if running on Colab and not yet logged into Hugging Face:
from huggingface_hub import login
login()

PyTorch version: 2.14.1+cu130
CUDA available: True
Device: Tesla T4


In [4]:
# 4. Run Baseline 1 & DiffMean using the original benchmark scripts (1-baseline.sh & 2-diffmean-4.sh)
# !bash 1-baseline.sh
!bash 2-diffmean-4.sh

bash: 2-diffmean-4.sh: No such file or directory


In [10]:
# 5. Evaluate Language Forcing Success Rate using the original evaluation scripts
!bash X-evaluate-language-forcing-baseline.sh
!bash X-evaluate-language-forcing-diffmean.sh

=== Evaluating Baseline 1 ===
Found 9 result files

LOADING AND PROCESSING DATA

Successfully processed 90 detections
Across 9 language pairs


PER-TARGET-LANGUAGE SUCCESS RATES
(How often was the model successfully forced to generate each language?)
Target Lang  Success Rate    Successes    Total Tests 
--------------------------------------------------------------------------------
de             100.0%         30           30          
en              96.7%         29           30          
es             100.0%         30           30          
--------------------------------------------------------------------------------
OVERALL         98.9%         89           90          

PER-SOURCE-LANGUAGE SUCCESS RATES
(How often did activation succeed starting from each language?)
Source Lang  Success Rate    Successes    Total Tests  Target Langs
--------------------------------------------------------------------------------
es             100.0%         30           30           3   

In [ ]:
# 6. Visualize Layer-by-Layer DiffMean Results (B-vis-ablate-diffmean.sh)
!bash B-vis-ablate-diffmean.sh

from IPython.display import Image, display
display(Image(filename="generation/diffmean/success_rate_by_layer.png"))